• Sentiment analysis
• Style analysis or comparison
• Authorship identification
• Topic modelling

External sources file URL:

1. Ghostbuster : https://www.scifiscripts.com/scripts/Ghostbusters.txt
2. Middlemarch : cleaned metadata from lab 4
3. Newspaper (EV Cars): https://www.cnn.com/interactive/2019/08/business/electric-cars-audi-volkswagen-tesla/, https://www.cnn.com/2019/08/01/cars/future-of-electric-car-charging/index.html, https://www.cnn.com/2019/08/07/business/ford-ceo-hackett-elon-musk-table-interview/index.html

**Importing Libraries**

In [1]:
import nltk
from nltk.corpus import names
# import the NLTK packages we know we need
from nltk.tokenize import word_tokenize
from nltk import FreqDist
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

import random
import numpy as np
import os
import pandas as pd

import re
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
import gensim
from gensim import corpora
from gensim.models import LdaModel
from nltk import punkt
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

import spacy
nlp = spacy.load("en_core_web_sm")
from spacy import displacy
from nltk.sentiment.vader import SentimentIntensityAnalyzer

C:\Users\sevy\anaconda3\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


**Sentiment analysis**

- Code from Lecture week 10: Sentiment Analysis was used for the assignment

In [3]:
def get_sentiment_scores(text):
    """
    Uses VADER within NLTK to calculate sentiment
    
    Args:
        text (str): a string containing the file or text
        
    Returns: 
        dict: a dictionary that VADER creates
    """
    analyzer = SentimentIntensityAnalyzer()
    score = analyzer.polarity_scores(text)
    return score

def process_dir(path):
    """
    Reads all the files in a directory. Processes them using the 'get_sentiment_scores' function
    
    Args: 
        path (str): path to the directory where the files are
        
    Returns:
        dict: a dictionary with file names and their sentiment scores
    
    """
    scores = {}

    for filename in os.listdir(path):
        if filename.endswith(".txt"):    
            file_path = os.path.join(path, filename)      
            with open(file_path, 'r', encoding="utf-8") as f:
                text = f.read()
                scores[filename] = get_sentiment_scores(text)
    return scores

In [4]:
path = './rawdata'
scores_files = process_dir(path)

In [5]:
print(scores_files)

{'EVCars.txt': {'neg': 0.042, 'neu': 0.868, 'pos': 0.091, 'compound': 0.9998}, 'Ghostbusters.txt': {'neg': 0.083, 'neu': 0.831, 'pos': 0.086, 'compound': 0.9982}, 'Middlemarch.txt': {'neg': 0.088, 'neu': 0.773, 'pos': 0.139, 'compound': 1.0}}


In [6]:
# of course, pandas makes everything look better
df = pd.DataFrame.from_dict(scores_files, orient="index")
df

,neg,neu,pos,compound
EVCars.txt,0.042,0.868,0.091,0.9998
Ghostbusters.txt,0.083,0.831,0.086,0.9982
Middlemarch.txt,0.088,0.773,0.139,1.0000


**Function used for Style analysis and Authorship Identification**

**style analysis or comparison**


**Authorship Identification**

**Topic Modelling**
- Code from Lecture 13: Topic Modelling was used for this assignment

In [9]:
### Function used to preprocess the data for topic modelling
# define a function that: 1. tokenizes, 2. lowercases and
# 3. lemmatizes

# create the lemmatizer first
lemmatizer = WordNetLemmatizer()

# define the function
def process_text(text):
    clean_text = re.sub(r'<.*?>', '', text)
    words = word_tokenize(clean_text.lower())
    cleaned_words = [word for word in words if word.isalpha() and word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word) for word in cleaned_words]
    return lemmatized_words

In [65]:
path = './rawdata'

**Ghostbuster**

In [89]:
file_path_g = os.path.join(path, 'Ghostbusters.txt')

In [91]:
with open(file_path_g, 'r', encoding = "utf-8") as f:
    text_ghost = f.read()

In [93]:
text_ghost_cleaned = process_text(text_ghost)

In [94]:
# use the gensim function to create a dictionary of the words in the text
dictionary_ghost = corpora.Dictionary([text_ghost_cleaned])

In [97]:
# you can inspect the contents of that dictionary
for token, token_id in list(dictionary_ghost.token2id.items())[:10]:
    print('{} => {}'.format(token, token_id))

aaaggghh => 0
aagghh => 1
abandoned => 2
ability => 3
absorbed => 4
absorbs => 5
abuse => 6
academic => 7
accelerator => 8
accent => 9


In [99]:
# use the doc2bow function to create a corpus, a bag of words (bow) of the text and the word counts
corpus_ghost = [dictionary_ghost.doc2bow(text_ghost_cleaned)]

In [101]:
# create the LDA model
lda_model_ghost = LdaModel(corpus_ghost, num_topics=10, id2word=dictionary_ghost, passes=15)

In [103]:
# print the top 20 words for each topic
topics_ghost = lda_model_ghost.print_topics(num_words=20)
for topic_num, topic in topics_ghost:
    print(f"Topic {topic_num}: ", end="")
    words = topic.split(' + ')
    word_list = [word.split('*')[1].strip('\"') for word in words] 
    print(", ".join(word_list))
    print()

Topic 0: venkman, stantz, spengler, dana, louis, door, look, int, ghostbusters, get, see, ext, one, night, peck, come, start, like, day, turn

Topic 1: venkman, dana, stantz, spengler, louis, look, door, int, ghostbusters, get, see, think, turn, one, come, winston, ext, back, start, right

Topic 2: venkman, stantz, spengler, dana, look, louis, door, ghostbusters, see, night, ext, int, get, right, one, start, think, building, like, turn

Topic 3: venkman, stantz, spengler, dana, look, louis, ghostbusters, door, get, int, one, janine, come, peck, back, like, night, think, building, winston

Topic 4: venkman, stantz, spengler, louis, door, dana, look, ghostbusters, int, get, back, like, right, around, start, peck, one, see, open, ext

Topic 5: venkman, spengler, stantz, dana, louis, look, door, int, peck, right, one, ghostbusters, like, ext, get, see, come, man, building, back

Topic 6: venkman, dana, stantz, spengler, louis, int, door, get, ghostbusters, peck, building, one, see, ext, lo

**Middlemarch**

In [106]:
file_path_m = os.path.join(path, 'Middlemarch.txt')

In [108]:
with open(file_path_m, 'r', encoding = "utf-8") as f:
    text_middle = f.read()

In [110]:
text_middle_cleaned = process_text(text_middle)

In [111]:
# use the gensim function to create a dictionary of the words in the text
dictionary_middle = corpora.Dictionary([text_middle_cleaned])

In [112]:
# you can inspect the contents of that dictionary
for token, token_id in list(dictionary_middle.token2id.items())[:10]:
    print('{} => {}'.format(token, token_id))

abandon => 0
abandoned => 1
abandoning => 2
abandonment => 3
abated => 4
abdicated => 5
abel => 6
aberration => 7
abeyance => 8
abide => 9


In [113]:
# use the doc2bow function to create a corpus, a bag of words (bow) of the text and the word counts
corpus_middle = [dictionary_middle.doc2bow(text_middle_cleaned)]

In [114]:
# create the LDA model
lda_model_middle = LdaModel(corpus_middle, num_topics=10, id2word=dictionary_middle, passes=15)

In [115]:
# print the top 20 words for each topic
topics = lda_model_middle.print_topics(num_words=20)
for topic_num, topic in topics:
    print(f"Topic {topic_num}: ", end="")
    words = topic.split(' + ')
    word_list = [word.split('*')[1].strip('\"') for word in words] 
    print(", ".join(word_list))
    print()

Topic 0: said, would, lydgate, could, dorothea, might, know, one, must, man, say, made, little, thing, casaubon, think, much, like, bulstrode, never

Topic 1: said, would, lydgate, one, could, dorothea, know, man, little, might, come, bulstrode, like, good, must, say, casaubon, fred, felt, think

Topic 2: said, would, dorothea, man, one, lydgate, casaubon, fred, know, like, might, see, little, could, must, never, bulstrode, made, rosamond, good

Topic 3: would, said, dorothea, lydgate, know, could, man, one, might, never, casaubon, like, thing, must, fred, thought, little, made, much, make

Topic 4: said, would, dorothea, could, lydgate, one, casaubon, little, bulstrode, rosamond, man, might, fred, see, know, much, must, think, made, make

Topic 5: said, would, lydgate, dorothea, one, could, know, like, think, must, casaubon, say, little, rosamond, might, never, fred, thing, good, man

Topic 6: said, would, lydgate, could, one, dorothea, know, like, rosamond, never, good, bulstrode, ca

**EVCars**

In [123]:
file_path_e = os.path.join(path, 'EVCars.txt')

In [125]:
with open(file_path_e, 'r', encoding = "utf-8") as f:
    text_ev = f.read()

In [127]:
text_ev_cleaned = process_text(text_ev)

In [129]:
# use the gensim function to create a dictionary of the words in the text
dictionary_ev = corpora.Dictionary([text_ev_cleaned])

In [131]:
# you can inspect the contents of that dictionary
for token, token_id in list(dictionary_ev.token2id.items())[:10]:
    print('{} => {}'.format(token, token_id))

ability => 0
able => 1
absolutely => 2
accident => 3
according => 4
account => 5
achieve => 6
acknowledges => 7
acquired => 8
acquisition => 9


In [133]:
# use the doc2bow function to create a corpus, a bag of words (bow) of the text and the word counts
corpus_ev = [dictionary_ev.doc2bow(text_ev_cleaned)]

In [135]:
# create the LDA model
lda_model_ev = LdaModel(corpus_ev, num_topics=10, id2word=dictionary_ev, passes=15)

In [137]:
# print the top 20 words for each topic
topics = lda_model_ev.print_topics(num_words=20)
for topic_num, topic in topics:
    print(f"Topic {topic_num}: ", end="")
    words = topic.split(' + ')
    word_list = [word.split('*')[1].strip('\"') for word in words] 
    print(", ".join(word_list))
    print()

Topic 0: car, electric, company, charger, tesla, vehicle, ford, said, volkswagen, year, charging, model, hackett, station, make, fast, also, industry, battery, customer

Topic 1: car, electric, charger, company, tesla, vehicle, said, ford, volkswagen, model, charging, year, also, station, battery, industry, network, fast, new, one

Topic 2: car, electric, charger, company, vehicle, ford, said, tesla, volkswagen, make, model, year, charging, hackett, station, fast, also, battery, customer, ceo

Topic 3: car, electric, tesla, company, volkswagen, year, said, vehicle, charger, charging, ford, model, battery, hackett, make, station, industry, production, fast, diesel

Topic 4: car, electric, company, volkswagen, tesla, charger, ford, vehicle, said, year, station, charging, fast, also, model, hackett, make, billion, new, industry

Topic 5: car, electric, tesla, charger, company, vehicle, ford, said, year, volkswagen, hackett, model, fast, make, charging, customer, station, audi, industry, a